# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayyk7/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Simple threshold heuristics are highly inefficient for scheduling content updates.
*   **Where the label comes from:** The label is an observed outcome proxy derived directly from Google Search Console (GSC) performance drops over a subsequent 30-day window (i.e., whether a page's daily impressions or traffic fell below its historical baseline).
*   **Does the validation design carry the claim?** Yes, but only under a grouped split. If evaluated on a simple random split, the validation falsely inflates performance because pages from the same client are highly correlated. Under a client-grouped split, we prove that a static threshold (like "refresh every 180 days") underperforms machine learning when transferring to completely unseen clients.

### Finding 2: High-impression "superstar" pages have a structurally higher likelihood of documented decay.
*   **Where the label comes from:** The label comes from the same GSC 30-day future performance drop indicator.
*   **Does the validation design carry the claim?** Yes. Because our validation split groups data by client, we can confidently show that across all clients—regardless of their industry or niche—high-exposure assets show a higher rate of flagged performance drop. This is structurally honest because the validation prevents the model from simply memorizing specific high-traffic client domains.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Methodology questions documented. Label sources and validation designs audited.")

Methodology questions documented. Label sources and validation designs audited.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### The Split Showdown: Random vs. Grouped Split

To demonstrate why a random row split is "dishonest," we run a head-to-head test:
1.  **Before (Dishonest Split):** We run a standard random train/test split. Because pages from the same client exist in both sets, the model overfits by memorizing client-specific search baselines.
2.  **After (Honest Split):** We run a `client_id` Grouped Split, ensuring that no client in the test set has ever been seen during training. This measures true real-world generalizability.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# 1. Clone repository in Google Colab to access data directory
if 'google.colab' in sys.modules:
    REPO_URL = "https://github.com/rafayyk7/flyrank-ml.git"
    REPO_DIR = "flyrank-ml"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository into Google Colab...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

# 2. Establish path and load the data
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../../data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(csv_path)

# Fill missing values with medians
features_list = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
for col in features_list:
    df[col] = df[col].fillna(df[col].median())

df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 3. Precision@50 Calculation Helper
def evaluate_p50(df_eval, score_col, target_col, k=50):
    sorted_df = df_eval.sort_values(by=score_col, ascending=False).head(k)
    return sorted_df[target_col].sum() / k

# --- NAIVE / DISHONEST SPLIT (Random Split) ---
train_rand, test_rand = train_test_split(df, test_size=0.2, random_state=42)

rf_naive = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_naive.fit(train_rand[features_list], train_rand['is_declining_label'])

test_rand = test_rand.copy()
test_rand['score'] = rf_naive.predict_proba(test_rand[features_list])[:, 1]
p50_naive = evaluate_p50(test_rand, 'score', 'is_declining_label')

# --- HONEST SPLIT (Client-Grouped Split) ---
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, df['is_declining_label'], groups=df['client_id']))
train_group = df.iloc[train_idx].copy()
test_group = df.iloc[test_idx].copy()

rf_honest = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_honest.fit(train_group[features_list], train_group['is_declining_label'])

test_group['score'] = rf_honest.predict_proba(test_group[features_list])[:, 1]
p50_honest = evaluate_p50(test_group, 'score', 'is_declining_label')

# Print comparison
print("--- SPLIT VALIDATION COMPARISON ---")
print(f"Naive (Random) Split Precision@50: {p50_naive * 100:.1f}%")
print(f"Honest (Grouped) Split Precision@50: {p50_honest * 100:.1f}%")
print(f"Difference: { (p50_honest - p50_naive) * 100:+.1f}% (Representing the honest generalizability drop)")

Cloning repository into Google Colab...
--- SPLIT VALIDATION COMPARISON ---
Naive (Random) Split Precision@50: 92.0%
Honest (Grouped) Split Precision@50: 56.0%
Difference: -36.0% (Representing the honest generalizability drop)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Final Feature Leakage Audit
We run a Pearson correlation check between our final feature vector and our target label to ensure no "future information" (like downstream manual flags or mathematical labels in disguise) has sneaked past our pipeline gates. Any correlation $|r| > 0.90$ is flagged immediately.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- Launching Final Leakage Hunt ---")

X_honest = test_group[features_list]
y_honest = test_group['is_declining_label']

correlations = X_honest.apply(lambda col: col.corr(y_honest))

leakage_found = False
for feature, corr_val in correlations.items():
    abs_corr = abs(corr_val)
    print(f"Correlation: '{feature:22}' with Target = {corr_val:+.4f}")
    if abs_corr > 0.90:
        print(f"🚨 LEAKAGE WARNING: '{feature}' has an extremely high correlation ({corr_val:.4f}) with target!")
        leakage_found = True

if not leakage_found:
    print("\n✅ Leakage Hunt Complete: No feature correlations exceed the 0.90 threshold. Final feature vector is completely clean!")

--- Launching Final Leakage Hunt ---
Correlation: 'content_age_days      ' with Target = -0.0894
Correlation: 'days_since_last_update' with Target = -0.0546
Correlation: 'impressions_90d       ' with Target = -0.0295
Correlation: 'avg_position          ' with Target = -0.0782
Correlation: 'ctr                   ' with Target = -0.0214
Correlation: 'word_count            ' with Target = +0.0195

✅ Leakage Hunt Complete: No feature correlations exceed the 0.90 threshold. Final feature vector is completely clean!


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### The Defensive Claim Audit

*   **Bold, Unsafe Claim (Do NOT say this):**
    > *"Our machine learning model successfully reverse-engineers Google's ranking algorithm, guaranteeing that any flagged page we rewrite will reclaim its top-ranking position and boost traffic."*

*   **Rewritten Safe Claim (DO say this):**
    > *"Our Random Forest model acts as a directional decision-support tool. Based on historical observations of search positions, CTR fluctuations, and content age, it ranks pages by their statistical likelihood of performance decay relative to their past baseline metrics. This allows content teams to prioritize their manual rewrite pipeline more efficiently."*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Claim Audit: Unsafe, causal claims successfully replaced with safe, decision-support claims.")

Claim Audit: Unsafe, causal claims successfully replaced with safe, decision-support claims.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.